In [1]:
!pip install -q chromadb sentence-transformers pymupdf openai anthropic transformers openpyxl

import os
import re
import json
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from openai import OpenAI
import anthropic
import chromadb
import fitz

print('tout est ok')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 906.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [ ]:
os.environ["OPENAI_API_KEY"] = "METTRE CLE API OPENAI ICI"
os.environ["DEEPSEEK_API_KEY"] = "METTRE CLE API DEEPSEEK ICI"
os.environ["ANTHROPIC_API_KEY"] = "METTRE CLE API ANTHROPIC ICI"

client_gpt = OpenAI(api_key="METTRE CLE API OPENAI ICI")
client_deepseek = OpenAI(api_key="METTRE CLE API DEEPSEEK ICI", base_url="https://api.deepseek.com")
client_claude = anthropic.Anthropic(api_key="METTRE CLE API ANTHROPIC ICI")

print('Loading BioBERT embeddings...')
os.environ["TOKENIZERS_PARALLELISM"] = "false"
embedder = SentenceTransformer("dmis-lab/biobert-base-cased-v1.2")
print('tout est ok')

chroma_client = chromadb.Client()
print('tout est ok')

ner = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")
print('tout est ok')

Loading BioBERT embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tout est ok


In [ ]:
def extraire_phrases_pdf(nom_fichier):
    """Extraire les phrases du PDF"""
    try:
        doc = fitz.open(nom_fichier)
        phrases = []
        for page in doc:
            texte = page.get_text()
            for ligne in texte.split('\n'):
                ligne = ligne.strip()
                if len(ligne) > 50:
                    phrases.append(ligne)
        return phrases
    except Exception as e:
        print(f'Erreur lecture {nom_fichier}: {e}')
        return []

print('tout est ok')

tout est ok


In [ ]:
def est_phrase_clinique(ligne):
    """Filtrer les phrases cliniquement pertinentes (regex + keywords)"""
    # Rejeter les références
    if re.search(r'\d{4};\d+:\d+', ligne):
        return False
    # Rejeter URLs
    if re.search(r'https?://', ligne):
        return False
    # Minimum longueur
    if len(ligne.split()) < 8:
        return False
    # Pas trop d'abréviations
    if len(re.findall(r'\b[A-Z]{2,6}\b', ligne)) > 5:
        return False
    # Au moins un mot-clé clinique
    mots_cles = ['is', 'are', 'recommended', 'associated', 'treatment',
                 'risk', 'should', 'patients', 'may', 'reduces', 'indicated',
                 'considered', 'hypertension', 'pressure', 'therapy', 'management']
    return any(m in ligne.lower() for m in mots_cles)

def contient_entites_biomedicales(phrase, ner_model, seuil=2):
    """Vérifier que la phrase a >= 2 entités biomédicales (NER)"""
    try:
        entites = ner_model(phrase[:512])
        return len(entites) >= seuil
    except:
        return False

print('tout est ok')

nickel


In [ ]:
def indexer_phrases_chromadb(phrases_filtrees, collection_name="esc_recommendations"):
    """
    Indexer les phrases filtrées dans ChromaDB avec BioBERT embeddings
    C'est la clé du RAG : on crée une base vectorielle des recommandations
    """

    # Créer/obtenir la collection
    try:
        chroma_client.delete_collection(name=collection_name)
    except:
        pass

    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    print(f'Indexing {len(phrases_filtrees)} phrases...')

    # Chunking en batches (ChromaDB a limite de 5000 par batch)
    batch_size = 500
    for i in range(0, len(phrases_filtrees), batch_size):
        batch = phrases_filtrees[i:i+batch_size]

        # Générer embeddings BioBERT
        embeddings = embedder.encode(batch).tolist()

        # Ajouter à ChromaDB
        collection.add(
            documents=batch,
            embeddings=embeddings,
            ids=[f'phrase_{i+j}' for j in range(len(batch))]
        )

        if (i + batch_size) % 2000 == 0:
            print(f'  ✓ Indexed {i + batch_size}/{len(phrases_filtrees)} phrases')

    print(f'nickel: {len(phrases_filtrees)} phrases')
    return collection

print('tout est ok')

nickel


In [ ]:
from google.colab import files

print('Upload your recommendation PDFs')
uploaded = files.upload()
pdf_files = list(uploaded.keys())
print(f'top {len(pdf_files)} PDFs uploaded')

# ========== EXTRACT PHRASES ==========
print('\n[STEP 1/3] Extracting phrases from PDFs...')
all_phrases = []

for pdf_file in pdf_files:
    phrases_brutes = extraire_phrases_pdf(pdf_file)
    all_phrases.extend(phrases_brutes)
    print(f'  ✓ {pdf_file}: {len(phrases_brutes)} raw phrases')

print(f'Total raw phrases: {len(all_phrases)}')

# ========== FILTER PHRASES ==========
print('\n[STEP 2/3] Filtering phrases (regex + keywords + NER >= 2 entities)...')
phrases_filtrees = [p for p in all_phrases if est_phrase_clinique(p)]
print(f'  ✓ After regex/keywords: {len(phrases_filtrees)} phrases')

phrases_ner = [p for p in phrases_filtrees if contient_entites_biomedicales(p, ner)]
print(f'  ✓ After NER (>= 2 entities): {len(phrases_ner)} phrases')

# ========== INDEX IN CHROMADB ==========
print('\n[STEP 3/3] Indexing in ChromaDB with BioBERT embeddings...')
collection = indexer_phrases_chromadb(phrases_ner, collection_name="esc_recommendations")

print(f'   Indexed phrases: {len(phrases_ner)}')

Upload your 2-3 ESC recommendation PDFs...


Saving ehad192.pdf to ehad192.pdf
Saving ehae176.pdf to ehae176.pdf
Saving ehae178.pdf to ehae178.pdf
top 3 PDFs uploaded

[STEP 1/3] Extracting phrases from PDFs...
  ✓ ehad192.pdf: 7010 raw phrases
  ✓ ehae176.pdf: 7634 raw phrases
  ✓ ehae178.pdf: 7599 raw phrases
Total raw phrases: 22243

[STEP 2/3] Filtering phrases (regex + keywords + NER >= 2 entities)...
  ✓ After regex/keywords: 10917 phrases
  ✓ After NER (>= 2 entities): 8886 phrases

[STEP 3/3] Indexing in ChromaDB with BioBERT embeddings...
Indexing 8886 phrases...
  ✓ Indexed 2000/8886 phrases
  ✓ Indexed 4000/8886 phrases
  ✓ Indexed 6000/8886 phrases
  ✓ Indexed 8000/8886 phrases
nickel: 8886 phrases
   Indexed phrases: 8886


In [7]:
SYNONYMES_NORMALIZATION = {
    'coronary artery disease': 'coronary artery disease',
    'cad': 'coronary artery disease',
    'ischemic heart disease': 'coronary artery disease',
    'ihd': 'coronary artery disease',
    'myocardial infarction': 'myocardial infarction',
    'mi': 'myocardial infarction',
    'ami': 'myocardial infarction',
    'heart attack': 'myocardial infarction',
    'acute coronary syndrome': 'acute coronary syndrome',
    'acs': 'acute coronary syndrome',
    'nstemi': 'acute coronary syndrome',
    'stemi': 'acute coronary syndrome',
    'heart failure': 'heart failure',
    'hf': 'heart failure',
    'chf': 'heart failure',
    'cardiac failure': 'heart failure',
    'atrial fibrillation': 'atrial fibrillation',
    'af': 'atrial fibrillation',
    'afib': 'atrial fibrillation',
    'hypertension': 'hypertension',
    'high blood pressure': 'hypertension',
    'elevated blood pressure': 'hypertension',
    'arterial hypertension': 'hypertension',
    'type 2 diabetes': 'type 2 diabetes',
    't2dm': 'type 2 diabetes',
    'diabetes mellitus': 'type 2 diabetes',
    'chronic kidney disease': 'chronic kidney disease',
    'ckd': 'chronic kidney disease',
    'renal impairment': 'chronic kidney disease',
    'kidney failure': 'chronic kidney disease',
    'dyslipidemia': 'dyslipidemia',
    'hyperlipidemia': 'dyslipidemia',
    'stroke': 'stroke',
    'cerebrovascular accident': 'stroke',
    'cva': 'stroke',
    'peripheral arterial disease': 'peripheral arterial disease',
    'pad': 'peripheral arterial disease',
    'lower-extremity artery disease': 'peripheral arterial disease',
    'lead': 'peripheral arterial disease',
    'heart failure with reduced ejection fraction': 'HFrEF',
    'hfref': 'HFrEF',
    'systolic heart failure': 'HFrEF',
    'heart failure with preserved ejection fraction': 'HFpEF',
    'hfpef': 'HFpEF',
    'diastolic heart failure': 'HFpEF',
    'heart failure with mildly reduced ejection fraction': 'HFmrEF',
    'hfmref': 'HFmrEF',
    'left ventricular hypertrophy': 'left ventricular hypertrophy',
    'lvh': 'left ventricular hypertrophy',
    'systolic dysfunction': 'systolic dysfunction',
    'diastolic dysfunction': 'diastolic dysfunction',
    'left atrial enlargement': 'left atrial enlargement',
    'la enlargement': 'left atrial enlargement',
    'SGLT2 inhibitor': 'SGLT2 inhibitor',
    'sglt2i': 'SGLT2 inhibitor',
    'sglt2': 'SGLT2 inhibitor',
    'GLP-1 receptor agonist': 'GLP-1 receptor agonist',
    'glp-1 ra': 'GLP-1 receptor agonist',
    'glp-1': 'GLP-1 receptor agonist',
    'glp1': 'GLP-1 receptor agonist',
    'insulin': 'insulin',
    'metformin': 'metformin',
    'ACE inhibitor': 'ACE inhibitor',
    'ace-i': 'ACE inhibitor',
    'ace inhibitors': 'ACE inhibitor',
    'ace-inhibitor': 'ACE inhibitor',
    'acei': 'ACE inhibitor',
    'ARB': 'ARB',
    'angiotensin receptor blocker': 'ARB',
    'angiotensin ii receptor blocker': 'ARB',
    'arb inhibitor': 'ARB',
    'ARNI': 'ARNI',
    'beta blocker': 'beta blocker',
    'beta-blocker': 'beta blocker',
    'statin': 'statin',
    'statins': 'statin',
    'diuretic': 'diuretic',
    'calcium channel blocker': 'calcium channel blocker',
    'ccb': 'calcium channel blocker',
    'aspirin': 'aspirin',
    'acetylsalicylic acid': 'aspirin',
    'warfarin': 'warfarin',
    'antiplatelet': 'antiplatelet',
    'dual antiplatelet therapy': 'dual antiplatelet therapy',
    'dapt': 'dual antiplatelet therapy',
    'echocardiography': 'echocardiography',
    'echo': 'echocardiography',
    'tte': 'echocardiography',
    'tee': 'echocardiography',
    'electrocardiogram': 'electrocardiogram',
    'ecg': 'electrocardiogram',
    'ekg': 'electrocardiogram',
    'coronary angiography': 'coronary angiography',
    'angiography': 'coronary angiography',
    'blood pressure monitoring': 'blood pressure monitoring',
    'bp monitoring': 'blood pressure monitoring',
    'holter monitoring': 'holter monitoring',
    'holter': 'holter monitoring',
    'stress test': 'stress test',
    'HbA1c': 'HbA1c',
    'hba1c': 'HbA1c',
    'glycated haemoglobin': 'HbA1c',
    'left ventricular ejection fraction': 'left ventricular ejection fraction',
    'lvef': 'left ventricular ejection fraction',
    'ejection fraction': 'left ventricular ejection fraction',
    'estimated glomerular filtration rate': 'estimated glomerular filtration rate',
    'egfr': 'estimated glomerular filtration rate',
    'gfr': 'estimated glomerular filtration rate',
    'natriuretic peptide': 'natriuretic peptide',
    'bnp': 'natriuretic peptide',
    'nt-probnp': 'natriuretic peptide',
    'nt-pro-bnp': 'natriuretic peptide',
    'b-type natriuretic peptide': 'natriuretic peptide',
    'albuminuria': 'albuminuria',
    'proteinuria': 'albuminuria',
    'microalbuminuria': 'albuminuria',
    'troponin': 'troponin',
    'creatinine': 'creatinine',
}

if __name__ == '__main__':
    print(f"Normalisation dict: {len(SYNONYMES_NORMALIZATION)} entries")

Normalisation dict: 118 entries


In [8]:
VALID_RELATIONS = {
    'est_facteur_de_risque_de': {'code': '405506007', 'desc': 'Has etiology / Risk factor'},
    'est_traite_par': {'code': '116681000007109', 'desc': 'Has therapeutic procedure'},
    'est_symptome_de': {'code': '363270007', 'desc': 'Related finding'},
    'est_complication_de': {'code': '408729004', 'desc': 'Has component'},
    'est_surveille_par': {'code': '363713009', 'desc': 'Has measurement'},
}

def normaliser_entite(entite_brute):
    entite_clean = entite_brute.lower().strip()
    return SYNONYMES_NORMALIZATION.get(entite_clean, entite_brute)

def deduplicater_triplets(triplets_list):
    seen = set()
    deduped = []
    for t in triplets_list:
        key = (t.get('sujet_normalise', '').lower(), t.get('relation', '').lower(), t.get('objet_normalise', '').lower())
        if key not in seen:
            seen.add(key)
            deduped.append(t)
    return deduped

def valider_triplet_symbolique(relation):
    return relation in VALID_RELATIONS, f"Relation {relation} valide" if relation in VALID_RELATIONS else f"Relation inconnue: {relation}"

def filtrer_triplets_valides(triplets_list):
    triplets_valides = []
    triplets_rejetes = []
    for t in triplets_list:
        if t.get('sujet_normalise', '').lower() == t.get('objet_normalise', '').lower():
            t['rejection_reason'] = "Auto-référentiel: sujet == objet"
            triplets_rejetes.append(t)
            continue
        is_valid, reason = valider_triplet_symbolique(t.get('relation', ''))
        if is_valid:
            triplets_valides.append(t)
        else:
            t['rejection_reason'] = reason
            triplets_rejetes.append(t)
    return triplets_valides, triplets_rejetes

def ajouter_snomed_uris(triplets):
    for t in triplets:
        relation = t.get('relation', '')
        if relation in VALID_RELATIONS:
            t['snomed_code'] = VALID_RELATIONS[relation]['code']
            t['snomed_uri'] = f"http://snomed.info/id/{VALID_RELATIONS[relation]['code']}"
    return triplets

print(' Validation & SNOMED functions defined')

 Validation & SNOMED functions defined


In [9]:
def calculer_consensus(triplets_par_llm):
    """Consensus >= 2/3 modèles"""
    triplets_normalized = {}

    for llm_name, triplets in triplets_par_llm.items():
        for t in triplets:
            if not t:
                continue
            key = (normaliser_entite(t.get('sujet', '')).lower(), t.get('relation', '').lower(), normaliser_entite(t.get('objet', '')).lower())
            if key not in triplets_normalized:
                triplets_normalized[key] = {
                    'sujet': normaliser_entite(t.get('sujet', '')),
                    'relation': t.get('relation', ''),
                    'objet': normaliser_entite(t.get('objet', '')),
                    'modeles_accord': [],
                    'sources': []
                }
            triplets_normalized[key]['modeles_accord'].append(llm_name)
            if 'source' in t:
                triplets_normalized[key]['sources'].append(t['source'])

    triplets_consensus = []
    for key, triplet_data in triplets_normalized.items():
        consensus_score = len(triplet_data['modeles_accord']) / 3
        if consensus_score >= 1/2:
            triplet_data['consensus'] = f"{len(triplet_data['modeles_accord'])}/3"
            triplets_consensus.append(triplet_data)

    return triplets_consensus

print(' Consensus function defined')

 Consensus function defined


In [10]:
def exporter_pour_review_hitl(triplets_consensus, filename="triplets_a_review_HITL.xlsx"):
    export_data = []
    for i, t in enumerate(triplets_consensus):
        export_data.append({
            'ID': i + 1,
            'Sujet': t.get('sujet', ''),
            'Relation': t.get('relation', ''),
            'Objet': t.get('objet', ''),
            'Consensus': t.get('consensus', ''),
            'Modèles': ', '.join(t.get('modeles_accord', [])),
            'SNOMED_Code': t.get('snomed_code', ''),
            'SNOMED_URI': t.get('snomed_uri', ''),
            'Validation_Manuelle': '',
        })
    df_export = pd.DataFrame(export_data)
    df_export.to_excel(filename, index=False, engine='openpyxl')
    print(f' HITL file: {filename}')
    return df_export

def exporter_rdf_snomed(triplets_finaux, filename="enrichissement_SNOMED.rdf"):
    rdf_content = '''<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
         xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"
         xmlns:owl="http://www.w3.org/2002/07/owl#"
         xmlns:cvdo="http://cvdo.org/"
         xmlns:snomed="http://snomed.info/id/">
  <!-- Enrichissement Ontologie Cardiovasculaire via RAG + LLM -->
'''
    # Mapping des codes SNOMED aux propriétés
    snomed_to_property = {
        '405506007': 'cvdo:hasRiskFactor',
        '116681000007109': 'cvdo:hasTreatment',
        '363270007': 'cvdo:hasSymptom',
        '408729004': 'cvdo:hasComplication',
        '363713009': 'cvdo:isMonitoredBy'
    }

    for t in triplets_finaux:
        sujet = t.get('sujet', '').replace(' ', '_')
        objet = t.get('objet', '').replace(' ', '_')
        snomed_code = t.get('snomed_code', '')
        property_name = snomed_to_property.get(snomed_code, 'cvdo:relatedTo')

        rdf_content += f'  <rdf:Description rdf:about="http://cvdo.org/{sujet}">\n'
        rdf_content += f'    <{property_name} rdf:resource="http://cvdo.org/{objet}"/>\n'
        rdf_content += f'    <cvdo:snomed_code>{snomed_code}</cvdo:snomed_code>\n'
        rdf_content += f'  </rdf:Description>\n'

    rdf_content += '</rdf:RDF>'
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(rdf_content)
    print(f'✓ RDF file: {filename}')

In [11]:
queries = [
    "hypertension causes risk factors etiology",
    "hypertension treatment antihypertensive drugs therapy",
    "hypertension complications organ damage cardiovascular",
    "blood pressure monitoring screening diagnosis detection",
    "resistant hypertension management refractory",
    "pregnancy hypertension preeclampsia gestational",
    "secondary hypertension causes aldosteronism pheochromocytoma",
    "lifestyle modifications diet exercise salt reduction hypertension",
    "elderly hypertension blood pressure targets frailty",
    "diabetes chronic kidney disease hypertension comorbidity",

    "atrial fibrillation risk factors causes obesity alcohol",
    "atrial fibrillation anticoagulation treatment oral anticoagulants",
    "atrial fibrillation complications stroke thromboembolism",
    "atrial fibrillation symptoms palpitations dyspnea fatigue",
    "catheter ablation pulmonary vein isolation rhythm control",
    "rate control strategy beta blocker digoxin atrial fibrillation",
    "atrial fibrillation monitoring echocardiography ECG screening",
    "CHA2DS2-VASc score stroke risk stratification anticoagulation",
    "atrial fibrillation heart failure reduced ejection fraction",
    "postoperative atrial fibrillation cardiac surgery prevention",

    "acute coronary syndrome risk factors smoking diabetes dyslipidemia",
    "myocardial infarction treatment reperfusion PCI thrombolysis",
    "acute coronary syndrome complications heart failure cardiogenic shock",
    "chest pain symptoms troponin diagnosis acute coronary syndrome",
    "dual antiplatelet therapy DAPT aspirin P2Y12 inhibitor stenting",
    "troponin BNP cardiac biomarkers monitoring prognosis",
    "STEMI NSTEMI classification management emergency",
    "coronary artery bypass grafting CABG revascularization surgery",
    "cardiac rehabilitation secondary prevention post myocardial infarction",
    "statin therapy lipid lowering cardiovascular prevention post ACS",
]
print(f' {len(queries)} semantic queries defined')

 30 semantic queries defined


In [12]:
# ============================================================
# FIX 5 — Amélioration du prompt pour est_symptome_de
# ============================================================
# PROBLÈME : La relation est_symptome_de ne produit qu'1 seul
# triplet sur 114. Les LLM ne la reconnaissent pas assez.
# Le prompt actuel ne donne aucun exemple ni guidance sur
# ce que constitue un symptôme vs un facteur de risque.
#
# SOLUTION : On enrichit le prompt avec des exemples explicites
# et des instructions de disambiguation.
# ============================================================

prompt_template = """You are a medical ontology expert specializing in cardiovascular diseases.

Your task is to extract semantic triplets from the clinical context below.

INSTRUCTIONS:
1. Extract ALL triplets that can be inferred from the provided context
2. Use ONLY information from the context, but interpret it flexibly
3. Allowed relations: est_facteur_de_risque_de, est_traite_par, est_symptome_de, est_complication_de, est_surveille_par
4. Format: subject -> relation -> object
5. Subject = medical entity (disease, drug, condition, symptom)
6. Object = target condition or outcome
7. If NO valid triplet can be extracted, reply: NO_TRIPLET

RELATION GUIDELINES:
- est_facteur_de_risque_de: A condition or factor that INCREASES THE PROBABILITY of another disease (e.g. obesity -> est_facteur_de_risque_de -> hypertension)
- est_traite_par: A disease/condition is treated by a drug or intervention. Subject MUST be the disease, object MUST be the treatment (e.g. hypertension -> est_traite_par -> ACE inhibitors)
- est_symptome_de: A clinical sign or symptom that MANIFESTS in a disease. Look for: chest pain, dyspnea, palpitations, headache, dizziness, fatigue, edema, syncope, nausea (e.g. chest pain -> est_symptome_de -> acute coronary syndrome, palpitations -> est_symptome_de -> atrial fibrillation)
- est_complication_de: A disease that RESULTS FROM another disease over time (e.g. stroke -> est_complication_de -> atrial fibrillation)
- est_surveille_par: A disease is monitored by a test or measurement (e.g. hypertension -> est_surveille_par -> ambulatory blood pressure monitoring)

ENTITY QUALITY RULES:
- Use SPECIFIC medical terms, not vague categories
- BAD: "cardiovascular events", "organ damage", "CVD", "medications"
- GOOD: "stroke", "left ventricular hypertrophy", "ACE inhibitors", "troponin"
- Subject and object must NEVER be the same entity

CLINICAL CONTEXT (source of truth):
{context}

Extract ALL triplets from this context.
Reply in English, one triplet per line.
No markdown, no bullets, no explanations.
Triplets:"""

print(' Prompt template loaded (v2 — enriched)')

 Prompt template loaded (v2 — enriched)


In [13]:
def pipeline_rag_llm_amelioree(queries, collection, num_queries_max=None):
    """
    Pipeline RAG + Multi-LLM avec:
    - RAG Retrieval (ChromaDB + BioBERT)
    - Multi-LLM Extraction (GPT-4o, DeepSeek, Claude)
    - Consensus Validation (>= 1/2)
    - Normalization (synonymes + dedup)
    - Symbolic Validation (règles)
    - SNOMED-CT Alignment
    """

    all_results = []
    queries_to_process = queries[:num_queries_max] if num_queries_max else queries

    for query_idx, query in enumerate(queries_to_process):
        print(f"\n{'='*70}")
        print(f"Query {query_idx + 1}/{len(queries_to_process)}: '{query}'")
        print(f"{'='*70}")

        # ========== RAG RETRIEVAL ==========
        print(f"  [1/4] RAG Retrieval (ChromaDB + BioBERT)...")
        query_embedding = embedder.encode([query]).tolist()
        retrieval_results = collection.query(
            query_embeddings=query_embedding,
            n_results=30
        )

        retrieved_docs = retrieval_results["documents"][0]
        retrieved_distances = retrieval_results["distances"][0]

        context = "\\n---\\n".join([
            f"[Source {i+1}, similarity: {1-d:.2f}] {doc}"
            for i, (doc, d) in enumerate(zip(retrieved_docs, retrieved_distances))
            if d < 0.7
        ])

        if not context:
            context = "\\n---\\n".join([
                f"[Source {i+1}] {doc}"
                for i, doc in enumerate(retrieved_docs[:5])
            ])

        print(f"     ✓ Retrieved {len(retrieved_docs)} documents from ChromaDB")

        # ========== MULTI-LLM EXTRACTION ==========
        print(f"  [2/4] Multi-LLM Extraction...")
        prompt = prompt_template.format(context=context)

        triplets_par_llm = {"gpt-4o": [], "deepseek": [], "claude": []}

        # GPT-4o
        try:
            r = client_gpt.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            output = r.choices[0].message.content
            for line in output.split("\n"):
                if " -> " in line and "NO_TRIPLET" not in line:
                    parts = line.split(" -> ")
                    if len(parts) == 3:
                        triplets_par_llm["gpt-4o"].append({
                            "sujet": parts[0].strip(),
                            "relation": parts[1].strip(),
                            "objet": parts[2].strip(),
                            "source": retrieved_docs[0] if retrieved_docs else "unknown"
                        })
            print(f"     ✓ GPT-4o: {len(triplets_par_llm['gpt-4o'])} triplets")
        except Exception as e:
            print(f"     ✗ GPT-4o error: {e}")

        # DeepSeek
        try:
            r = client_deepseek.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            output = r.choices[0].message.content
            for line in output.split("\n"):
                if " -> " in line and "NO_TRIPLET" not in line:
                    parts = line.split(" -> ")
                    if len(parts) == 3:
                        triplets_par_llm["deepseek"].append({
                            "sujet": parts[0].strip(),
                            "relation": parts[1].strip(),
                            "objet": parts[2].strip(),
                            "source": retrieved_docs[0] if retrieved_docs else "unknown"
                        })
            print(f"     ✓ DeepSeek: {len(triplets_par_llm['deepseek'])} triplets")
        except Exception as e:
            print(f"     ✗ DeepSeek error: {e}")

        # Claude
        try:
            r = client_claude.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1000,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            output = r.content[0].text
            for line in output.split("\n"):
                if " -> " in line and "NO_TRIPLET" not in line:
                    parts = line.split(" -> ")
                    if len(parts) == 3:
                        triplets_par_llm["claude"].append({
                            "sujet": parts[0].strip(),
                            "relation": parts[1].strip(),
                            "objet": parts[2].strip(),
                            "source": retrieved_docs[0] if retrieved_docs else "unknown"
                        })
            print(f"     ✓ Claude: {len(triplets_par_llm['claude'])} triplets")
        except Exception as e:
            print(f"     ✗ Claude error: {e}")

        # ========== CONSENSUS + NORMALIZATION + VALIDATION ==========
        print(f"  [3/4] Post-processing...")

        triplets_consensus = calculer_consensus(triplets_par_llm)
        print(f"     ✓ Consensus (>= 2/3): {len(triplets_consensus)} triplets")

        for t in triplets_consensus:
            t["sujet_normalise"] = normaliser_entite(t["sujet"])
            t["objet_normalise"] = normaliser_entite(t["objet"])

        triplets_consensus = deduplicater_triplets(triplets_consensus)
        print(f"     ✓ After dedup: {len(triplets_consensus)} triplets")

        triplets_valides, triplets_rejetes = filtrer_triplets_valides(triplets_consensus)
        print(f"     ✓ Symbolic validation: {len(triplets_valides)} valid, {len(triplets_rejetes)} rejected")

        # ========== ADD SNOMED URIs ==========
        print(f"  [4/4] Adding SNOMED-CT URIs...")
        triplets_valides = ajouter_snomed_uris(triplets_valides)
        print(f"     ✓ SNOMED URIs added")

        all_results.extend(triplets_valides)

    return all_results

print(' Main pipeline function defined')

 Main pipeline function defined


In [15]:
triplets_finaux = pipeline_rag_llm_amelioree(queries, collection, num_queries_max=30)

print(f"\n PIPELINE COMPLETED")
print(f"   Total final triplets: {len(triplets_finaux)}")


Query 1/30: 'hypertension causes risk factors etiology'
  [1/4] RAG Retrieval (ChromaDB + BioBERT)...
     ✓ Retrieved 30 documents from ChromaDB
  [2/4] Multi-LLM Extraction...
     ✓ GPT-4o: 4 triplets
     ✓ DeepSeek: 0 triplets
     ✓ Claude: 9 triplets
  [3/4] Post-processing...
     ✓ Consensus (>= 2/3): 1 triplets
     ✓ After dedup: 1 triplets
     ✓ Symbolic validation: 1 valid, 0 rejected
  [4/4] Adding SNOMED-CT URIs...
     ✓ SNOMED URIs added

Query 2/30: 'hypertension treatment antihypertensive drugs therapy'
  [1/4] RAG Retrieval (ChromaDB + BioBERT)...
     ✓ Retrieved 30 documents from ChromaDB
  [2/4] Multi-LLM Extraction...
     ✓ GPT-4o: 8 triplets
     ✓ DeepSeek: 19 triplets
     ✓ Claude: 10 triplets
  [3/4] Post-processing...
     ✓ Consensus (>= 2/3): 10 triplets
     ✓ After dedup: 10 triplets
     ✓ Symbolic validation: 10 valid, 0 rejected
  [4/4] Adding SNOMED-CT URIs...
     ✓ SNOMED URIs added

Query 3/30: 'hypertension complications organ damage cardiov

In [16]:
#Après la pipeline, on dédoublonne sur le tuple (sujet_normalise, relation, objet_normalise) en gardant le triplet avec le meilleur score de consensus.

avant = len(triplets_finaux)

# Dictionnaire : clé = (sujet_norm, relation, objet_norm) -> meilleur triplet
seen = {}
for t in triplets_finaux:
    key = (
        t["sujet_normalise"].strip().lower(),
        t["relation"].strip().lower(),
        t["objet_normalise"].strip().lower()
    )
    if key not in seen:
        seen[key] = t
    else:
        # Garder celui avec le meilleur consensus
        score_existant = int(seen[key]["consensus"].split("/")[0])
        score_nouveau = int(t["consensus"].split("/")[0])
        if score_nouveau > score_existant:
            seen[key] = t

triplets_finaux = list(seen.values())

apres = len(triplets_finaux)
print(f"  Avant  : {avant} triplets")
print(f"  Après  : {apres} triplets")
print(f"  Supprimés : {avant - apres} doublons")

  Avant  : 204 triplets
  Après  : 137 triplets
  Supprimés : 67 doublons


In [17]:
# On définit une liste de termes connus comme traitements/médicaments. Si le sujet est un traitement et l'objet est une maladie, on inverse.

# Termes identifiés comme traitements / médicaments / interventions
TERMES_TRAITEMENTS = {
    'ace inhibitor', 'ace inhibitors',
    'arbs', 'arb',
    'beta blockers', 'beta blocker',
    'calcium channel blockers', 'calcium channel blocker',
    'diuretic', 'diuretics', 'loop diuretic', 'loop diuretics',
    'thiazide', 'thiazide diuretic',
    'spironolactone', 'finerenone', 'amiodarone',
    'sglt2 inhibitors', 'sglt2 inhibitor',
    'antihypertensive agents', 'antihypertensive medications',
    'antihypertensive treatments', 'antihypertensive therapy',
    'bp-lowering medications',
    'angiotensin ii receptor blockade',
    'renal denervation',
    'catheter ablation',
    'coronary artery bypass grafting',
    'pci', 'thrombolysis', 'reperfusion',
    'statin', 'statins', 'statin therapy',
    'aspirin', 'anticoagulation', 'oral anticoagulants',
    'dual antiplatelet therapy', 'dapt',
    'single-pill combination therapy',
    'triple combination antihypertensive medication',
    'regular aerobic exercise', 'lifestyle modifications',
    'blocker treatment',
    'cardiac rehabilitation',
}

inversions = 0
for t in triplets_finaux:
    if t["relation"] == "est_traite_par":
        sujet_lower = t["sujet_normalise"].strip().lower()
        objet_lower = t["objet_normalise"].strip().lower()
        # Si le sujet est un traitement et l'objet ne l'est pas -> inverser
        if sujet_lower in TERMES_TRAITEMENTS and objet_lower not in TERMES_TRAITEMENTS:
            # Inverser sujet <-> objet
            t["sujet"], t["objet"] = t["objet"], t["sujet"]
            t["sujet_normalise"], t["objet_normalise"] = t["objet_normalise"], t["sujet_normalise"]
            inversions += 1

print(f"  Inversions corrigées : {inversions}")

  Inversions corrigées : 0


In [18]:
# On recalcule le consensus en dédupliquant les modèles (set), puis on met à jour le score.

import ast

corriges = 0
for t in triplets_finaux:
    modeles_raw = t.get("modeles_accord", [])
    # Si c'est une string (lu depuis CSV), la convertir en liste
    if isinstance(modeles_raw, str):
        try:
            modeles_raw = ast.literal_eval(modeles_raw)
        except:
            continue

    modeles_uniques = list(set(modeles_raw))
    nouveau_score = f"{len(modeles_uniques)}/3"
    ancien_score = t.get("consensus", "")

    if ancien_score != nouveau_score:
        corriges += 1

    t["modeles_accord"] = modeles_uniques
    t["consensus"] = nouveau_score

print(f"  Scores corrigés : {corriges}")
print(f"  Distribution finale :")
from collections import Counter
scores = Counter(t["consensus"] for t in triplets_finaux)
for score, nb in sorted(scores.items()):
    print(f"    {score} : {nb} triplets")

  Scores corrigés : 14
  Distribution finale :
    1/3 : 9 triplets
    2/3 : 88 triplets
    3/3 : 40 triplets


In [22]:


print("\n" + "="*70)
print("FIX 4bis — Filtre triplets sous le seuil de consensus")
print("="*70)

avant = len(triplets_finaux)
triplets_finaux = [t for t in triplets_finaux if int(t["consensus"].split("/")[0]) >= 2]
apres = len(triplets_finaux)

print(f"  Avant : {avant} triplets")
print(f"  Après : {apres} triplets")
print(f"  Supprimés (consensus < 2/3) : {avant - apres}")


FIX 4bis — Filtre triplets sous le seuil de consensus
  Avant : 135 triplets
  Après : 126 triplets
  Supprimés (consensus < 2/3) : 9


In [23]:
#On filtre les triplets dont le sujet ou l'objet fait partie d'une liste de termes vagues, et ceux



TERMES_VAGUES = {
    'cvd events', 'cardiovascular events', 'cv events',
    'cardiovascular risk', 'cvd risk', 'cv risk',
    'cvd', 'organ damage', 'target organ damage',
    'blocker treatment', 'medications', 'treatments',
    'outcomes', 'risk', 'events', 'disease',
    'bp-lowering medications',
    'antihypertensive medications',
    'antihypertensive agents',
    'antihypertensive treatments',
}

avant = len(triplets_finaux)
triplets_propres = []
supprimes_vagues = 0
supprimes_auto = 0

for t in triplets_finaux:
    s = t["sujet_normalise"].strip().lower()
    o = t["objet_normalise"].strip().lower()

    # Filtrer auto-référentiels
    if s == o:
        supprimes_auto += 1
        continue

    # Filtrer entités vagues
    if s in TERMES_VAGUES or o in TERMES_VAGUES:
        supprimes_vagues += 1
        continue

    triplets_propres.append(t)

triplets_finaux = triplets_propres

print(f"  Nombre de triplet finaux  : {len(triplets_finaux)} triplets")

  Nombre de triplet finaux  : 126 triplets


In [24]:

# CSV
df_final = pd.DataFrame(triplets_finaux)
df_final.to_csv("XP2_triplets_consensus_normalises.csv", index=False, encoding="utf-8")
print(" CSV: XP2_triplets_consensus_normalises.csv")

# RDF/OWL
#exporter_rdf_snomed(triplets_finaux, "XP2_enrichissement_SNOMED.rdf")

print("\n" + "="*70)
print(" PIPELINE COMPLETE WITH SUCCESS!")
print("="*70)
print("\nGenerated files:")
print("  1. XP2_triplets_consensus_normalises.csv")
print("  2. XP2_triplets_a_review_HITL.xlsx (for expert validation)")
print("  3. XP2_enrichissement_SNOMED.rdf (CVDO enrichment)")
print("\nNext steps:")
print("  - Open XLSX for manual validation (HITL)")
print("  - Fill 'Validation_Manuelle' column (✅/❌)")
print("  - Use RDF for CVDO integration")

 CSV: XP2_triplets_consensus_normalises.csv

 PIPELINE COMPLETE WITH SUCCESS!

Generated files:
  1. XP2_triplets_consensus_normalises.csv
  2. XP2_triplets_a_review_HITL.xlsx (for expert validation)
  3. XP2_enrichissement_SNOMED.rdf (CVDO enrichment)

Next steps:
  - Open XLSX for manual validation (HITL)
  - Fill 'Validation_Manuelle' column (✅/❌)
  - Use RDF for CVDO integration


In [21]:
from google.colab import files
import os

print("\n" + "="*70)
print(" TÉLÉCHARGEMENT DES FICHIERS GÉNÉRÉS")
print("="*70 + "\n")

# Vérifier quels fichiers existent
output_files = {
    "CSV (Quick Analysis)": "XP2_triplets_consensus_normalises.csv"}

files_to_download = []

for nom, fichier in output_files.items():
    if os.path.exists(fichier):
        files_to_download.append(fichier)
        size_mb = os.path.getsize(fichier) / (1024*1024)
        print(f" {nom}: {fichier} ({size_mb:.2f} MB)")
    else:
        print(f"❌ {nom}: {fichier} (NOT FOUND)")

# Télécharger les fichiers
if files_to_download:
    print(f"\n Downloading {len(files_to_download)} files...\n")
    for fichier in files_to_download:
        files.download(fichier)
        print(f"✓ Downloaded: {fichier}")
else:
    print("\n⚠️ No files to download. Check if pipeline has run successfully.")

print("\n" + "="*70)
print(" DOWNLOAD COMPLETE!")
print("="*70)
print("\nFiles downloaded:")
print("  1. XP2_triplets_consensus_normalises.csv")
print("     → Open in Excel/Sheets to see all triplets")
print("     → Columns: sujet, relation, objet, consensus, modeles, snomed_uri")
print("\n  2. XP2_enrichissement_SNOMED.rdf")
print("     → RDF/OWL format for ontology integration")

# Vérifier combien de triplets ont été générés en mémoire
print(f"Nombre total de triplets en mémoire: {len(triplets_finaux)}")



 TÉLÉCHARGEMENT DES FICHIERS GÉNÉRÉS

 CSV (Quick Analysis): XP2_triplets_consensus_normalises.csv (0.13 MB)




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: XP2_triplets_consensus_normalises.csv

 DOWNLOAD COMPLETE!

Files downloaded:
  1. XP2_triplets_consensus_normalises.csv
     → Open in Excel/Sheets to see all triplets
     → Columns: sujet, relation, objet, consensus, modeles, snomed_uri

  2. XP2_enrichissement_SNOMED.rdf
     → RDF/OWL format for ontology integration
Nombre total de triplets en mémoire: 135
